# TODOs

- add online data...

In [1]:
from shared.main import * 

In [ ]:
[c for c in data_online.columns if 'first' in c]

# trial-wise relationship state under different weighting rules

In [ ]:
# ============================================================
# Last-choice state: only the immediately previous choice
# ============================================================

def last_choice_state(choices):
    """
    State on trial t = choice made on trial t - 1.

    This is equivalent to a fixed sliding window of size 1.

    Parameters
    ----------
    choices : array, shape (n_trials, 2)

    Returns
    -------
    states : array, shape (n_trials, 2)
        First trial is NaN because there is no prior history.
    """
    choices = np.asarray(choices, dtype=float)
    states = np.full_like(choices, np.nan, dtype=float)

    states[1:] = choices[:-1]

    return states

# ============================================================
# Fixed sliding window: mean of the K most recent choices
# ============================================================

def sliding_window_state(choices, window_size=3):
    """
    State on trial t = mean of the most recent previous choices,
    up to a fixed window size.

    Before the full window is available, all available previous
    choices are used. For example, with window_size=3:

        trial 1 uses choice 0
        trial 2 uses choices 0:2
        trial 3 uses choices 0:3
        trial 4 uses choices 1:4
        ...

    Parameters
    ----------
    choices : array, shape (n_trials, 2)
    window_size : int
        Number of previous character-specific interactions to include.

    Returns
    -------
    states : array, shape (n_trials, 2)
        First trial is NaN because there is no prior history.
    """
    choices = np.asarray(choices, dtype=float)

    if window_size < 1 or int(window_size) != window_size:
        raise ValueError("window_size must be a positive integer.")

    window_size = int(window_size)
    states = np.full_like(choices, np.nan, dtype=float)

    for t in range(1, len(choices)):
        start = max(0, t - window_size)
        states[t] = choices[start:t].mean(axis=0)

    return states

# ============================================================
# Recency weighting: exponentially downweight older choices
# ============================================================

def recency_state(choices, half_life=2.0):
    """
    State on trial t = recency-weighted mean of previous choices.

    half_life is measured in character-specific interactions.
    A memory that is `half_life` interactions older receives
    half as much weight.

    The immediately preceding choice has age 0.

    Parameters
    ----------
    choices : array, shape (n_trials, 2)
    half_life : float
        Positive half-life of the exponential weighting function.

    Returns
    -------
    states : array, shape (n_trials, 2)
        First trial is NaN because there is no prior history.
    """
    choices = np.asarray(choices, dtype=float)

    if half_life <= 0:
        raise ValueError("half_life must be greater than zero.")

    states = np.full_like(choices, np.nan, dtype=float)

    for t in range(1, len(choices)):

        past = choices[:t]

        # Age 0 = immediately preceding interaction.
        ages = (t - 1) - np.arange(t)

        weights = 0.5 ** (ages / half_life)
        weights /= weights.sum()

        states[t] = np.sum(
            weights[:, None] * past,
            axis=0,
        )

    return states

# ============================================================
# Cumulative sum: original accumulating trajectory
# ============================================================

def cumulative_sum_state(choices):
    """
    State on trial t = sum of all choices before trial t.

    Unlike the running mean, this representation preserves the
    amount of accumulated history. Its magnitude can therefore
    increase over interactions.

    Parameters
    ----------
    choices : array, shape (n_trials, 2)

    Returns
    -------
    states : array, shape (n_trials, 2)
        First trial is NaN because there is no prior history.
    """
    choices = np.asarray(choices, dtype=float)
    states = np.full_like(choices, np.nan, dtype=float)

    for t in range(1, len(choices)):
        states[t] = choices[:t].sum(axis=0)

    return states

# ============================================================
# Uniform weighting: running mean of all previous choices
# ============================================================

def uniform_state(choices):
    """
    State on trial t = mean of all choices before trial t.

    Parameters
    ----------
    choices : array, shape (n_trials, 2)

    Returns
    -------
    states : array, shape (n_trials, 2)
        First trial is NaN because there is no prior history.
    """
    choices = np.asarray(choices, dtype=float)
    states = np.full_like(choices, np.nan, dtype=float)

    for t in range(1, len(choices)):
        states[t] = choices[:t].mean(axis=0)

    return states

# ============================================================
# Normalized mean direction: retain direction, discard magnitude
# ============================================================

def normalized_mean_direction_state(choices, eps=1e-12):
    """
    State on trial t = unit-normalized mean direction of all
    choices before trial t.

    This discards the magnitude of the running mean and retains
    only its direction.

    If previous choices cancel exactly, the state is set to [0, 0],
    indicating that there is no dominant historical direction.

    Parameters
    ----------
    choices : array, shape (n_trials, 2)
    eps : float
        Norm threshold below which the mean is treated as zero.

    Returns
    -------
    states : array, shape (n_trials, 2)
        First trial is NaN because there is no prior history.
    """
    choices = np.asarray(choices, dtype=float)
    states = np.full_like(choices, np.nan, dtype=float)

    for t in range(1, len(choices)):

        mean_state = choices[:t].mean(axis=0)
        magnitude = np.linalg.norm(mean_state)

        if magnitude > eps:
            states[t] = mean_state / magnitude
        else:
            states[t] = np.zeros(2, dtype=float)

    return states

# ============================================================
# Similarity weighting: retrieve choices similar to current
# ============================================================

def similarity_state(choices, beta=2.0):
    """
    State on trial t = similarity-weighted mean of previous choices.

    Current choice is the query.
    Previous choices are memories.

    Because choices are unit 2D direction vectors, the dot product is:

        +1 = same direction
         0 = orthogonal dimension
        -1 = opposite direction

    beta controls retrieval selectivity:

        beta = 0    -> uniform weighting
        larger beta -> stronger preference for similar choices

    Parameters
    ----------
    choices : array, shape (n_trials, 2)
    beta : float
        Inverse-temperature parameter controlling retrieval selectivity.

    Returns
    -------
    states : array, shape (n_trials, 2)
        First trial is NaN because there is no prior history.
    """
    choices = np.asarray(choices, dtype=float)
    states = np.full_like(choices, np.nan, dtype=float)

    for t in range(1, len(choices)):

        current = choices[t]
        past = choices[:t]

        # Dot product equals cosine similarity for these unit vectors.
        similarities = past @ current

        # Stable softmax.
        logits = beta * similarities
        logits -= logits.max()

        weights = np.exp(logits)
        weights /= weights.sum()

        states[t] = np.sum(
            weights[:, None] * past,
            axis=0,
        )

    return states

# ============================================================
# Apply any state function to a collection of choice arrays
# ============================================================

def build_states(choice_arrays, state_func, **kwargs):
    """
    Apply a state-building function independently to each choice array.

    Parameters
    ----------
    choice_arrays : iterable of arrays
        Each array should have shape (n_trials, 2).

        For example, choices_2d_by_char with shape (5, 12, 2)
        works directly because iterating over it gives five
        arrays with shape (12, 2).

    state_func : callable
        A function that accepts one (n_trials, 2) choice array and
        returns one (n_trials, 2) state array.

    **kwargs
        Extra arguments passed to state_func.

    Returns
    -------
    list of arrays
        One (n_trials, 2) state array per input choice array.
    """
    return [
        state_func(choices, **kwargs)
        for choices in choice_arrays
    ]

# ============================================================
# Model definitions
#
# Their order here determines their order in subject_states
# and in the plots.
# ============================================================

state_specs = [
    (
        "Last choice",
        last_choice_state,
        {},
    ),
    (
        "Sliding window (k=3)",
        sliding_window_state,
        {"window_size": 3},
    ),
    (
        "Recency weighted (half-life=2)",
        recency_state,
        {"half_life": 2.0},
    ),
    (
        "Cumulative sum",
        cumulative_sum_state,
        {},
    ),
    (
        "Uniform mean",
        uniform_state,
        {},
    ),
    (
        "Normalized mean direction",
        normalized_mean_direction_state,
        {},
    ),
    (
        "Similarity weighted (beta=2)",
        similarity_state,
        {"beta": 2.0},
    ),
]

state_names = [
    state_name
    for state_name, _, _ in state_specs
]


# ============================================================
# Build subject-level states
#
# Each subject_states value has shape:
#
#     n_models x 60 concatenated trials x 2 dimensions
#
# With the current specifications:
#
#     7 x 60 x 2
# ============================================================


subject_states = {}
subjective_placements = {}

for sub_id in incl_subs:

    behavior_df = load_behavior(sub_id)

    choices_2d = behavior_df[
        ["affil_decision", "power_decision"]
    ].values

    choices_2d_by_char = organize_by_character(
        choices_2d
    )  # 5 characters x 12 interactions x 2 dimensions

    # Build each state independently within each character.
    states_by_model = [
        build_states(
            choices_2d_by_char,
            state_func,
            **state_kwargs,
        )
        for _, state_func, state_kwargs in state_specs
    ]

    # For each model, concatenate the five character-specific
    # state trajectories into a single 60 x 2 array.
    subject_states[sub_id] = np.stack([
        np.concatenate(model_states, axis=0)
        for model_states in states_by_model
    ])

    # Retain the five final subjective placements for later
    # model comparisons.
    sub_data = data[
        data["sub_id"] == sub_id
    ].iloc[0]

    subjective_placements[sub_id] = np.asarray([
        [
            sub_data[f"affil_subj_{ch}"],
            sub_data[f"power_subj_{ch}"],
        ]
        for ch in CHARACTERS[:5]
    ])


# ============================================================
# Trial-pair distances
#
# For each subject and metric, the distance array has shape:
#
#     n_models x 60 trials x 60 trials
#
# Rows and columns corresponding to each character's first
# interaction are NaN because no preceding history exists.
# For cosine distance, zero-vector states are also NaN because
# their direction is undefined.
# ============================================================

distance_metrics = ("euclidean", "cosine")

def trial_pair_distance_matrix(
    representation,
    metric="euclidean",
    eps=1e-12,
):
    """Compute a trial-by-trial Euclidean or cosine distance matrix."""
    metric = str(metric).lower()
    if metric not in {"euclidean", "cosine"}:
        raise ValueError(
            "metric must be either 'euclidean' or 'cosine'."
        )

    representation = np.asarray(representation, dtype=float)
    valid_trials = np.all(np.isfinite(representation), axis=1)

    if metric == "cosine":
        # Cosine distance is undefined for zero-length vectors.
        valid_trials &= np.linalg.norm(representation, axis=1) > eps

    valid_states = representation[valid_trials]

    if metric == "euclidean":
        valid_distances = np.linalg.norm(
            valid_states[:, None, :] - valid_states[None, :, :],
            axis=-1,
        )
    else:
        unit_states = valid_states / np.linalg.norm(
            valid_states,
            axis=1,
            keepdims=True,
        )
        similarities = np.clip(unit_states @ unit_states.T, -1.0, 1.0)
        valid_distances = np.clip(1.0 - similarities, 0.0, 2.0)
        np.fill_diagonal(valid_distances, 0.0)

    distances = np.full(
        (len(representation), len(representation)),
        np.nan,
        dtype=float,
    )
    distances[np.ix_(valid_trials, valid_trials)] = valid_distances

    return distances

trial_pair_distances = {
    sub_id: {
        metric: np.stack([
            trial_pair_distance_matrix(
                representation,
                metric=metric,
            )
            for representation in states
        ])
        for metric in distance_metrics
    }
    for sub_id, states in subject_states.items()
}


# ============================================================
# Subject-level correlations between distance representations
# ============================================================

def correlate_distance_matrices(matrix_a, matrix_b, eps=1e-12):
    """Pearson correlation between matching finite upper triangles."""
    upper_triangle = np.triu_indices_from(matrix_a, k=1)
    values_a = matrix_a[upper_triangle]
    values_b = matrix_b[upper_triangle]

    valid_pairs = np.isfinite(values_a) & np.isfinite(values_b)
    values_a = values_a[valid_pairs]
    values_b = values_b[valid_pairs]
    n_trial_pairs = valid_pairs.sum()

    if (
        n_trial_pairs < 3
        or np.std(values_a) <= eps
        or np.std(values_b) <= eps
    ):
        return np.nan, n_trial_pairs

    return pearsonr(values_a, values_b)[0], n_trial_pairs


state_pairs = list(itertools.combinations(range(len(state_names)), 2))
correlation_records = []

for sub_id, distances_by_metric in trial_pair_distances.items():
    for metric, distance_matrices in distances_by_metric.items():
        for state_a_idx, state_b_idx in state_pairs:
            correlation, n_trial_pairs = correlate_distance_matrices(
                distance_matrices[state_a_idx],
                distance_matrices[state_b_idx],
            )
            state_a = state_names[state_a_idx]
            state_b = state_names[state_b_idx]
            correlation_records.append({
                "sub_id": sub_id,
                "distance_metric": metric,
                "state_a": state_a,
                "state_b": state_b,
                "comparison": f"{state_a} vs. {state_b}",
                "correlation": correlation,
                "n_trial_pairs": n_trial_pairs,
            })

subject_state_correlations = pd.DataFrame(correlation_records)

# ============================================================
# Plotting information
# ============================================================

distance_cmap = plt.get_cmap("viridis").copy()
distance_cmap.set_bad("white")

n_trials = next(iter(subject_states.values())).shape[1]
n_characters = len(CHARACTERS[:5])
n_trials_per_character = n_trials // n_characters

character_centers = (
    np.arange(n_characters) * n_trials_per_character
    + (n_trials_per_character - 1) / 2
)

character_boundaries = (
    np.arange(
        n_trials_per_character,
        n_trials,
        n_trials_per_character,
    )
    - 0.5
)


# ============================================================
# Plot one figure per subject
# ============================================================

for sub_id, distances_by_metric in trial_pair_distances.items():

    n_models = len(state_names)

    fig, axes = plt.subplots(
        len(distance_metrics),
        n_models,
        figsize=(3.5 * n_models, 9),
        constrained_layout=True,
    )

    axes = np.asarray(axes).reshape(len(distance_metrics), n_models)

    for metric_idx, metric in enumerate(distance_metrics):
        distance_matrices = distances_by_metric[metric]

        for model_idx, (state_name, distance_matrix) in enumerate(zip(
            state_names,
            distance_matrices,
        )):
            ax = axes[metric_idx, model_idx]
            ax.imshow(
                distance_matrix,
                cmap=distance_cmap,
                interpolation="none",
            )

            for boundary in character_boundaries:
                ax.axvline(
                    boundary,
                    color="white",
                    linewidth=0.6,
                    alpha=0.8,
                )
                ax.axhline(
                    boundary,
                    color="white",
                    linewidth=0.6,
                    alpha=0.8,
                )

            ax.set_xticks(character_centers)
            ax.set_xticklabels(
                CHARACTERS[:5],
                rotation=35,
                ha="right",
            )
            ax.set_yticks(character_centers)
            ax.set_yticklabels(CHARACTERS[:5])

            if metric_idx == 0:
                ax.set_title(state_name)
            if metric_idx == len(distance_metrics) - 1:
                ax.set_xlabel("Trial, grouped by character")

        axes[metric_idx, 0].set_ylabel(
            f"{metric.title()} distance\nTrial, grouped by character"
        )

    fig.suptitle(
        f"{sub_id}: trial-pair distances by state representation"
    )

    plt.show()
    plt.close(fig)

In [ ]:
# Distribution of subject-level correlations for every state comparison
comparison_order = [
    f'{state_names[state_a_idx]} vs. {state_names[state_b_idx]}'
    for state_a_idx, state_b_idx in state_pairs
]
metric_colors = {
    'euclidean': '#4C72B0',
    'cosine': '#DD8452',
}

fig, axes = plt.subplots(
    len(distance_metrics),
    len(comparison_order),
    figsize=(2.8 * len(comparison_order), 7),
    constrained_layout=True,
)
axes = np.asarray(axes).reshape(len(distance_metrics), len(comparison_order))

wilcoxon_records = []

for metric_idx, metric in enumerate(distance_metrics):
    for comparison_idx, comparison in enumerate(comparison_order):
        ax = axes[metric_idx, comparison_idx]
        comparison_df = subject_state_correlations[
            (subject_state_correlations['distance_metric'] == metric)
            & (subject_state_correlations['comparison'] == comparison)
        ]
        correlations = comparison_df['correlation'].dropna().to_numpy()

        sns.histplot(
            correlations,
            bins=12,
            binrange=(-1, 1),
            color=metric_colors[metric],
            edgecolor='white',
            ax=ax,
        )
        ax.axvline(0, color='black', linestyle='--', linewidth=1)
        ax.set_xlim(-1, 1)

        if len(correlations) == 0:
            statistic, p_value = np.nan, np.nan
        elif np.allclose(correlations, 0):
            statistic, p_value = 0.0, 1.0
        else:
            statistic, p_value = wilcoxon(
                correlations,
                alternative='greater',
            )

        if not np.isfinite(p_value):
            significance = 'n/a'
        elif p_value < 0.001:
            significance = '***'
        elif p_value < 0.01:
            significance = '**'
        elif p_value < 0.05:
            significance = '*'
        else:
            significance = 'ns'

        wilcoxon_records.append({
            'distance_metric': metric,
            'comparison': comparison,
            'n_subjects': len(correlations),
            'median_correlation': (
                np.nanmedian(correlations) if len(correlations) else np.nan
            ),
            'wilcoxon_statistic': statistic,
            'p_value_greater_than_zero': p_value,
            'significance': significance,
        })

        state_a = comparison_df['state_a'].iloc[0]
        state_b = comparison_df['state_b'].iloc[0]
        ax.set_title(f'{state_a}\nvs.\n{state_b}', fontsize=8)
        ax.text(
            0.04,
            0.96,
            f'W={statistic:.1f}\np={p_value:.3g} ({significance})\nn={len(correlations)}',
            transform=ax.transAxes,
            ha='left',
            va='top',
            fontsize=7,
        )

        ax.set_xlabel('Correlation (r)' if metric_idx == 1 else '')
        ax.set_ylabel('Participants' if comparison_idx == 0 else '')

    axes[metric_idx, 0].annotate(
        metric.title(),
        xy=(-0.55, 0.5),
        xycoords='axes fraction',
        rotation=90,
        ha='center',
        va='center',
        fontsize=12,
        fontweight='bold',
    )

correlation_wilcoxon_results = pd.DataFrame(wilcoxon_records)

fig.suptitle(
    'Subject-level Pearson correlations between state RDMs\n'
    'One-sided Wilcoxon signed-rank tests: correlation > 0',
)
plt.show()